# Pelajaran 12 - Pengurangan Sejarah Sembang dengan Draf Ejen

Buku nota ini menunjukkan cara mengurus konteks dalam perbualan panjang menggunakan Rangka Kerja Ejen Microsoft. Apabila perbualan berkembang, bilangan token meningkat — akhirnya melebihi tetingkap konteks model. Kami mengatasi ini dengan **corak ringkasan konteks** dan **draf ejen** untuk memori berterusan.

## Apa yang Akan Anda Pelajari:
1. **Kenapa Pengurusan Konteks Penting**: Memahami had token dan tetingkap konteks
2. **Ejen Sadar Konteks**: Membangun ejen yang mengurus konteks perbualan mereka sendiri
3. **Corak Ringkasan Konteks**: Menggunakan alat untuk memampatkan sejarah perbualan
4. **Draf Ejen**: Memori berterusan yang bertahan selepas pengurangan konteks

## Prasyarat:
- Persediaan Azure OpenAI dengan pembolehubah persekitaran yang dikonfigurasikan
- Kefahaman asas konsep ejen dari pelajaran sebelum ini


## Persediaan


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Mengapa Pengurusan Konteks Penting

Setiap LLM mempunyai **tingkap konteks** terhad — jumlah maksimum token yang boleh diproses dalam satu permintaan. Apabila perbualan multi-lawan berlangsung:

- **Bilangan token meningkat secara linear** dengan setiap mesej pengguna dan balasan pembantu.
- **Token prompt mendominasi kos** kerana keseluruhan sejarah dihantar semula setiap giliran.
- Akhirnya perbualan **melebihi tingkap konteks** dan model sama ada memendekkan atau menghasilkan ralat.

### Strategi untuk Mengurus Konteks

| Strategi | Cara Ia Berfungsi | Pertukaran |
|---|---|---|
| **Pemendekan** | Buang mesej tertua | Hilang konteks awal |
| **Penerangan ringkas** | Memadatkan mesej lama menjadi ringkasan | Sebahagian butiran hilang, tetapi titik utama dikekalkan |
| **Scratchpad / Memori Luaran** | Simpan fakta utama di luar perbualan | Memerlukan panggilan alat, tetapi bertahan walaupun pengurangan |

Dalam buku nota ini kami menggabungkan **penerangan ringkas** dengan **alat scratchpad** supaya agen boleh mengekalkan kesinambungan walaupun sejarah perbualan dipadatkan.


## Mewujudkan Ejen Berjaga Pada Konteks


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Mensimulasikan Perbualan Panjang

Mari kita lalui perbualan berbilang pusingan untuk melihat bagaimana konteks terkumpul. Ejen harus mengekalkan butiran penting (keutamaan, bajet, tarikh perjalanan) sepanjang pusingan dan menunjukkan kesinambungan.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Perhatikan bagaimana ejen mengekalkan konteks daripada pusingan-pusingan sebelumnya — ia tahu tentang Jepun, sushi, kuil, fotografi, bajet $3000, perjalanan solo, dan tempoh April. Dalam perbualan ringkas ini ia berfungsi dengan baik, tetapi apabila perbualan berkembang, sejarah penuh menjadi mahal untuk dihantar semula.

Mari teruskan perbualan dengan lebih banyak pusingan untuk melihat pengumpulan konteks:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Corak Ringkasan Konteks

Apabila perbualan berkembang, kita boleh menggunakan **alat ringkasan** untuk memadatkan konteks yang terkumpul ke dalam format yang padat. Ejen memanggil alat ini untuk merekod keutamaan utama supaya walaupun mesej lama dibuang, maklumat penting masih terpelihara.

Corak ini adalah blok binaan untuk pengurangan sejarah yang lebih sofistikated:
1. Ejen mengenal pasti fakta penting dari perbualan
2. Ia memanggil alat ringkasan untuk menyimpannya
3. Mesej lama boleh dibuang dengan selamat kerana ringkasan menangkap apa yang penting

Di bawah, kami mentakrifkan alat `summarize_preferences` yang boleh dipanggil oleh ejen untuk merekod ringkasan padat perkara yang telah dipelajarinya.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Ringkasan

Dalam pelajaran ini anda telah belajar cara mengurus konteks dalam perbualan agen jangka panjang menggunakan Microsoft Agent Framework:

### Konsep Utama
- **Tetingkap konteks adalah terhad** — setiap token dalam sejarah perbualan memerlukan kos dan dikira ke arah had.
- **Alat ringkasan** membolehkan agen memadatkan konteks terkumpul ke dalam ringkasan padat, mengurangkan penggunaan token sambil mengekalkan maklumat penting.
- **Pad skrap agen** menyediakan memori luaran berterusan yang kekal walaupun setelah pengurangan perbualan.

### Apa Yang Anda Bina
- **Agen sedar konteks** yang mengekalkan kesinambungan di sepanjang perbualan pelbagai giliran
- **Alat ringkasan** (`summarize_preferences`) yang merekodkan butiran utama pengguna dalam format padat
- **Perbualan pelbagai giliran** yang menunjukkan pengekalan dan pengendalian perubahan konteks

### Aplikasi Dunia Sebenar
- **Bot Perkhidmatan Pelanggan**: Ingat keutamaan sepanjang sesi sokongan yang panjang
- **Pembantu Peribadi**: Jejaki projek yang sedang berjalan tanpa perlu menerangkan semula konteks
- **Tutor Pendidikan**: Mengekalkan kemajuan pelajar di sepanjang banyak interaksi

### Langkah Seterusnya
- Laksanakan alat pad skrap penuh dengan ketekalan berasaskan fail
- Tambah pemangkasan sejarah automatik selepas ringkasan dibuat
- Gabungkan dengan pangkalan data vektor untuk carian memori semantik
- Bina agen yang boleh menyambung semula perbualan beberapa hari kemudian dengan konteks penuh


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
